### SetUp

In [ ]:
# # load API key from .env 
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# llm model
model="gpt-4o-mini"

### Business Logic

In [ ]:
from langchain_classic.chains import SequentialChain
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from langchain_openai import ChatOpenAI

# Langchain Model setup
llm = ChatOpenAI(temperature=0.85, model=model)

# Step 1: Poem Generator (Enhanced with optional famous references)
poem_prompt = ChatPromptTemplate.from_template(
    "Compose a concise 4–6 line poem on the theme: {theme}.\n"
    "If a well-known poem or poetic idea closely relates, you may subtly echo or reference it.\n"
    "Ensure originality while maintaining poetic elegance, rhythm, and imagery.\n"
    "Return only the poem without titles or extra formatting."
)
chain_poem = LLMChain(
    llm=llm,
    prompt=poem_prompt,
    output_key="poem"
)

# Step 2: Motivational Quote Generator (Enhanced with famous quotes)
quote_prompt = ChatPromptTemplate.from_template(
    "Generate a powerful motivational one-line quote about the theme: {theme}.\n"
    "If relevant, you may include or adapt a famous quote from a distinguished figure "
    "(e.g., philosophers, leaders, spiritual teachers), ensuring clarity and impact.\n"
    "Return only a single concise quote without explanation."
)
chain_quote = LLMChain(
    llm=llm,
    prompt=quote_prompt,
    output_key="quote"
)

# Step 3: Affirmation Generator (Refined tone)
affirm_prompt = ChatPromptTemplate.from_template(
    "Create a deeply empowering affirmation based on the theme: {theme}.\n"
    "It must begin with 'I am' or 'I can', and should feel personal, uplifting, and actionable.\n"
    "Keep it concise and emotionally resonant. Return only the affirmation."
)
chain_affirm = LLMChain(
    llm=llm,
    prompt=affirm_prompt,
    output_key="affirmation"
)

# Step 4: Reflection Prompt Generator (More thought-provoking)
reflect_prompt = ChatPromptTemplate.from_template(
    "Craft a meaningful journaling or self-reflection question inspired by the theme: {theme}.\n"
    "The question should encourage introspection, personal growth, and deeper awareness.\n"
    "Return only one clear question without explanation."
)
chain_reflect = LLMChain(
    llm=llm,
    prompt=reflect_prompt,
    output_key="reflection"
)

# Step 5: Motivational Guru Insight Chain (NEW)
guru_prompt = ChatPromptTemplate.from_template(
    "You are a wise and compassionate motivational guru who blends philosophy, history, and spirituality.\n"
    "Drawing inspiration from great thinkers (e.g., Buddha, Swami Vivekananda, Marcus Aurelius), "
    "historical events, and timeless wisdom, provide a short, powerful paragraph of guidance.\n\n"
    "Theme: {theme}\n\n"
    "Your response should:\n"
    "- Offer deep insight with a tone of hope, resilience, and compassion\n"
    "- Include a subtle reference to philosophy, history, spirituality, or a notable figure/event\n"
    "- Feel like timeless advice that uplifts and grounds the reader\n"
    "- Be concise (4–6 sentences max)\n\n"
    "Return only the paragraph without headings or extra formatting."
)
chain_guru = LLMChain(
    llm=llm,
    prompt=guru_prompt,
    output_key="guru_insight"
)

# Overall MindSpark Chain (Updated)
mindspark_chain = SequentialChain(
    chains=[
        chain_poem,
        chain_quote,
        chain_affirm,
        chain_reflect,
        chain_guru
    ],
    input_variables=["theme"],
    output_variables=[
        "poem",
        "quote",
        "affirmation",
        "reflection",
        "guru_insight"
    ],
    verbose=True
)

### Display Functions

In [ ]:
# Display helpers for Jupyter
from IPython.display import display, Markdown

def format_mindspark_markdown(response, theme):
    return f"""
### 🌟 MindSpark Output — Theme: {theme}

**Poem**
{response.get('poem', '')}

**Quote**  
{response.get('quote', '')}

**Affirmation**  
{response.get('affirmation', '')}

**Reflection Question**  
{response.get('reflection', '')}

**Guru Insight**  
{response.get('Insight', '')}
"""

# --- 8. Query ---
theme = "hope"
response = mindspark_chain({"theme": theme})

# --- 9. Display ---
md_text = format_mindspark_markdown(response, theme)
display(Markdown(md_text))


### UI

In [ ]:
# Add UI for Display
import ipywidgets as widgets
from IPython.display import display

# Chat display area
chat_area = widgets.HTML(value="", layout=widgets.Layout(width="100%", height="300px", overflow="auto", border="1px solid gray", padding="10px"))

# Input + send button
input_box = widgets.Text(placeholder="Enter a theme (e.g., courage, hope)...")
send_button = widgets.Button(description="Generate", button_style="success")

# Function to handle sending
def on_send(_):
    theme = input_box.value.strip()
    if not theme:
        return
    input_box.value = ""

    # Append user message
    chat_area.value += f"<p><b>You:</b> {theme}</p>"

    # Run MindSpark chain
    result = mindspark_chain({"theme": theme})

    # Format bot reply
    bot_reply = f"""
    <p><b>Bot (MindSpark):</b></p>
    <p><b>📜 Poem:</b><br>{result['poem']}</p>
    <p><b>💡 Quote:</b><br>{result['quote']}</p>
    <p><b>✨ Affirmation:</b><br>{result['affirmation']}</p>
    <p><b>🪞 Reflection:</b><br>{result['reflection']}</p>
    <p><b>🪞 Insight:</b><br>{result['guru_insight']}</p>
    """
    chat_area.value += bot_reply

# Bind button click
send_button.on_click(on_send)

# Reset button
reset_button = widgets.Button(description="Reset", button_style="warning")

def on_reset(_):
    chat_area.value = ""
    chat_area.value += "<p><b>Bot (MindSpark):</b> Welcome! Enter a theme to receive your inspiration pack.</p>"

reset_button.on_click(on_reset)

# Layout
ui = widgets.VBox([
    chat_area,
    widgets.HBox([input_box, send_button, reset_button])
])

# Initial greeting
chat_area.value += "<p><b>Bot (MindSpark):</b> Welcome! Enter a theme to receive your inspiration pack.</p>"

display(ui)
